In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import cv2
from glob import glob
import itertools
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID" 
os.environ["CUDA_VISIBLE_DEVICES"] = ""
from keras.utils import np_utils
from keras.preprocessing.image import ImageDataGenerator
from keras.models import Sequential
from keras.layers import Dense
from keras.layers import Dropout
from keras.layers import Flatten
from keras.layers.convolutional import Conv2D
from keras.layers.convolutional import MaxPooling2D
from keras.layers import BatchNormalization
from keras.callbacks import ModelCheckpoint,ReduceLROnPlateau,CSVLogger
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from tensorflow.python.framework import ops
ops.reset_default_graph()

Using TensorFlow backend.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Read the images and generate the train and test dataset (10 points)

In [3]:
os.listdir("/content/drive/My Drive/train")

['Fat Hen',
 'Maize',
 'Loose Silky-bent',
 'Small-flowered Cranesbill',
 'Common wheat',
 'Scentless Mayweed',
 'Common Chickweed',
 'Shepherds Purse',
 'Black-grass',
 'Charlock',
 'Sugar beet',
 'Cleavers']

In [0]:
train = "/content/drive/My Drive/train/*/*.png"
test = "/content/drive/My Drive/test/*.png"

In [0]:
train_files = glob(train)
test_files = glob(test)

In [0]:
train_images = []
train_label = []
for img in train_files:
    images=cv2.imread(img)
    train_images.append(cv2.resize(images, (70,70,)))
    train_label.append(img.split('/')[-2])

train_images = np.asarray(train_images)
train_label = pd.DataFrame(train_label)

In [7]:
print("The shape of train images is:",train_images.shape)
print(("The shape of train labels is:",train_label.shape))

The shape of train images is: (4764, 70, 70, 3)
('The shape of train labels is:', (4764, 1))


In [0]:
test_images = []
test_label = []
for img in test_files:
    images=cv2.imread(img)
    test_images.append(cv2.resize(images, (70,70,)))
    test_label.append(img.split('/')[-2])

test_images = np.asarray(test_images)
test_label = pd.DataFrame(test_label)

In [9]:
print("The shape of test images is:",test_images.shape)
print(("The shape of test labels is:",test_label.shape))

The shape of test images is: (794, 70, 70, 3)
('The shape of test labels is:', (794, 1))


In [0]:
train_images = train_images/255

In [11]:
label_encoder = preprocessing.LabelEncoder()
label_encoder.fit(train_label[0])
encoded_train_labels = label_encoder.transform(train_label[0])
encoded_labels = np_utils.to_categorical(encoded_train_labels)
classes = encoded_labels.shape[0]
print('Classes'+str(label_encoder.classes_))
print(str(classes))

Classes['Black-grass' 'Charlock' 'Cleavers' 'Common Chickweed' 'Common wheat'
 'Fat Hen' 'Loose Silky-bent' 'Maize' 'Scentless Mayweed'
 'Shepherds Purse' 'Small-flowered Cranesbill' 'Sugar beet']
4764


# Divide the data set into Train and validation data sets

In [0]:
X_train, X_val, y_train, y_val = train_test_split(train_images, encoded_labels, test_size=0.1, random_state=7, stratify = encoded_labels)

# Initialize & build the model (20 points)

In [13]:
import tensorflow as tf
tf.keras.backend.clear_session()
model = Sequential()

# Optimize the model (16 points)

In [14]:
model.add(Conv2D(32,(3, 3), input_shape=(70,70,3),activation= 'relu'))
model.add(Conv2D(32,(3, 3),activation= 'relu'))
model.add(MaxPooling2D())
model.add(Flatten())
model.add(Dense(512, activation= 'relu'))
model.add(Dense(12, activation='softmax'))

In [15]:
model.compile(loss = 'categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [16]:
print(model.summary())

Model: "sequential_1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv2d_1 (Conv2D)            (None, 68, 68, 32)        896       
_________________________________________________________________
conv2d_2 (Conv2D)            (None, 66, 66, 32)        9248      
_________________________________________________________________
max_pooling2d_1 (MaxPooling2 (None, 33, 33, 32)        0         
_________________________________________________________________
flatten_1 (Flatten)          (None, 34848)             0         
_________________________________________________________________
dense_1 (Dense)              (None, 512)               17842688  
_________________________________________________________________
dense_2 (Dense)              (None, 12)                6156      
Total params: 17,858,988
Trainable params: 17,858,988
Non-trainable params: 0
__________________________________________

In [17]:
model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs= 10, batch_size=10)

Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where



Train on 4287 samples, validate on 477 samples
Epoch 1/10





4287/4287 [==============================] - 174s 41ms/step - loss: 1.8806 - acc: 0.3599 - val_loss: 1.3343 - val_acc: 0.5367
Epoch 2/10
4287/4287 [==============================] - 173s 40ms/step - loss: 1.1120 - acc: 0.6160 - val_loss: 1.0144 - val_acc: 0.6583
Epoch 3/10
4287/4287 [==============================] - 173s 40ms/step - loss: 0.8089 - acc: 0.7294 - val_loss: 0.8443 - val_acc: 0.7065
Epoch 4/10
4287/4287 [==============================] - 174s 41ms/step - loss: 0.5549 - acc: 0.8171 - val_loss: 0.9431 - val_acc: 0.6918
Epoch 5/10
4287/4287 [==============================] - 173s 40ms/step - loss: 0.3523 - acc: 0.8845 - val_loss: 1.0198 - val_acc: 0.7044
Epoch 6/10
4287/4287 [==============================] - 173s 40ms/step - loss: 0.2367 - acc: 0.9221 - val_loss: 1.0476 - val_acc: 0.7254
Epoch 7/10
4287/4287 [=======

# Predict the accuracy for both train and validation data (14 points)

In [18]:
loss_and_metrics_1 = model.evaluate(X_train, y_train)
print(loss_and_metrics_1)

4287/4287 [==============================] - 18s 4ms/step
[0.05381549559690011, 0.9853044086773968]


In [19]:
loss_and_metrics_2 = model.evaluate(X_val, y_val)
print(loss_and_metrics_2)

477/477 [==============================] - 2s 4ms/step
[1.611343657945437, 0.7358490571036029]
